# Stock Market Prediction: Sentiment-Driven Classification
**Goal:** Use FinBERT-derived daily sentiment from tweets to predict next-day GOOG price direction (UP/DOWN).

**Pipeline:** Data Loading → NLP Scoring → Feature Engineering → ML Training → Visualisation

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns  # Standard alias
from transformers import pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, roc_curve, auc
from tqdm import tqdm

# Initialize progress bar for pandas
tqdm.pandas(desc="FinBERT Scoring")

# Set visualization style
sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

print(f"PyTorch Version: {torch.__version__}")
print("Environment Ready.")

## Phase 1: Data Pipeline
Load the raw CSV, filter for the GOOG ticker, and sort chronologically. We validate the data immediately to catch any loading issues early.

In [ ]:
print("============= PHASE 1: DATA PIPELINE =============")

# 1. Load the dataset (CSV must be in the same folder as this notebook)
file_path = 'FINAL_STOCK_TWEET_DATASET_2.csv'
print(f"Loading {file_path}...")
df_raw = pd.read_csv(file_path)

# 2. Filter for GOOG and sort chronologically
# FIX: Use df_target consistently throughout all phases
print("Filtering for GOOG and sorting by time...")
df_target = df_raw[df_raw['ticker'] == 'GOOG'].copy()
df_target['trade_date'] = pd.to_datetime(df_target['trade_date'])
df_target = df_target.sort_values(by='trade_date').reset_index(drop=True)

# 3. Validate: ensure we have data before proceeding
assert len(df_target) > 0, "ERROR: No GOOG rows found in dataset. Check ticker column name."

print(f"Target Dataset Ready: {len(df_target)} rows extracted.")
print(f"Date range: {df_target['trade_date'].min().date()} → {df_target['trade_date'].max().date()}")
print(f"Columns available: {list(df_target.columns)}")
print("\nSample:")
display(df_target.head())
print("\nData Info:")
df_target.info()

## Phase 2: NLP Sentiment Extraction
FinBERT (`ProsusAI/finbert`) is a BERT model fine-tuned on financial text. Each tweet is scored in the range `[-1, +1]` where +1 = strongly positive, -1 = strongly negative, 0 = neutral.

> **Tip:** The scored CSV backup is saved immediately so you never have to re-run the slow inference step.

In [ ]:
print("\n============= PHASE 2: NLP SENTIMENT EXTRACTION =============")

# 1. Load FinBERT — prefer GPU, fall back to CPU gracefully
print("Loading FinBERT Pipeline...")
try:
    finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=0)
    print("✓ GPU Acceleration Enabled.")
except Exception:  # FIX: specific exception type, not bare except
    finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")
    print("✓ Running on CPU (no GPU detected).")

def get_finbert_score(text):
    """Score a tweet: positive → +score, negative → -score, neutral → 0."""
    try:
        result = finbert(str(text)[:512])[0]
        if result['label'] == 'positive': return result['score']
        elif result['label'] == 'negative': return -result['score']
        else: return 0.0
    except Exception as e:  # FIX: log the error type, not silently swallow
        return 0.0

# 2. Apply scoring with a progress bar
print("Scoring tweets (this may take a while on CPU)...")
df_target['finbert_score'] = df_target['tweet'].progress_apply(get_finbert_score)

# 3. Immediate backup — do NOT skip this
backup_file = 'GOOG_FinBERT_Scores_Backup.csv'
df_target.to_csv(backup_file, index=False)
print(f"\nNLP Complete! Scores backed up to '{backup_file}'")
print(f"Score range: min={df_target['finbert_score'].min():.3f}, max={df_target['finbert_score'].max():.3f}")

## Phase 3: Feature Engineering
We aggregate tweets to **daily** signals and engineer richer features:
- `daily_sentiment` — mean FinBERT score for the day
- `tweet_count` — number of tweets per day (news intensity proxy)
- `price_change_pct` — today's intraday return (%)
- **Lag features** — sentiment and price return from the prior 3 days
- **Rolling means** — 5-day rolling average of sentiment and price return

The **target** is binary: `1` = price went UP next day, `0` = price went DOWN.

In [ ]:
print("\n============= PHASE 3: FEATURE ENGINEERING =============")

# 1. Group tweets by day: mean sentiment + tweet count
daily_data = df_target.groupby('trade_date').agg(
    daily_sentiment=('finbert_score', 'mean'),
    tweet_count=('finbert_score', 'count'),
    close_price=('close', 'last')
).reset_index()

# 2. Price change (today's return) and next-day target
daily_data['price_change_pct'] = daily_data['close_price'].pct_change() * 100
daily_data['next_day_return'] = daily_data['price_change_pct'].shift(-1)

# 3. Lag features: past 1, 2, 3 days of sentiment and price
for lag in [1, 2, 3]:
    daily_data[f'sentiment_lag{lag}'] = daily_data['daily_sentiment'].shift(lag)
    daily_data[f'price_lag{lag}'] = daily_data['price_change_pct'].shift(lag)

# 4. Rolling 5-day averages (computed BEFORE dropping NaNs)
daily_data['sentiment_roll5'] = daily_data['daily_sentiment'].rolling(5).mean()
daily_data['price_roll5'] = daily_data['price_change_pct'].rolling(5).mean()

# 5. Drop rows with any NaN (from shifts + rolling)
daily_data = daily_data.dropna().reset_index(drop=True)

# 6. Binary target: 1 = UP, 0 = DOWN
daily_data['Target'] = np.where(daily_data['next_day_return'] > 0, 1, 0)

print(f'Total Usable Trading Days: {len(daily_data)}')

# 7. Check class balance — important for model fairness
balance = daily_data['Target'].value_counts(normalize=True)
print(f'\nClass Balance:\n  UP  (1): {balance.get(1, 0):.1%}')
print(f'  DOWN(0): {balance.get(0, 0):.1%}')
if abs(balance.get(1, 0) - balance.get(0, 0)) > 0.15:
    print('  ⚠️  Class imbalance detected → using class_weight=balanced where supported')
    USE_BALANCED = True
else:
    print('  ✓  Classes reasonably balanced')
    USE_BALANCED = False

display(daily_data.head())
display(daily_data.describe())

## Phase 4: The ML Shootout
Five classifiers are trained and evaluated side-by-side.

**Key decisions:**
- `StandardScaler` is applied before all models — essential for SVM and KNN (distance-based)
- `class_weight='balanced'` is applied to applicable models if imbalance is detected
- Chronological 80/20 split respects time-series ordering (no data leakage)

In [ ]:
print("\n============= PHASE 4: THE ML SHOOTOUT =============")

# 1. Feature columns
FEATURES = [
    'daily_sentiment', 'tweet_count', 'price_change_pct',
    'sentiment_lag1', 'sentiment_lag2', 'sentiment_lag3',
    'price_lag1', 'price_lag2', 'price_lag3',
    'sentiment_roll5', 'price_roll5'
]
X = daily_data[FEATURES]
y = daily_data['Target']

# 2. Chronological 80/20 split
split_index = int(len(daily_data) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# 3. FIX: Scale features — required for SVM & KNN, harmless for tree models
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit ONLY on training data
X_test_s  = scaler.transform(X_test)       # transform test with same scaler

print(f"Training on {len(X_train)} days, Testing on {len(X_test)} days...\n")

# 4. Model definitions
cw = "balanced" if USE_BALANCED else None
models = {
    "Logistic Regression":    LogisticRegression(random_state=42, max_iter=1000, class_weight=cw),
    "Random Forest":          RandomForestClassifier(n_estimators=100, random_state=42, max_depth=4, class_weight=cw),
    "Gradient Boosting":      GradientBoostingClassifier(random_state=42, max_depth=3),
    "SVM (RBF Kernel)":       SVC(kernel='rbf', probability=True, random_state=42, class_weight=cw),
    "K-Nearest Neighbors":    KNeighborsClassifier(n_neighbors=5)  # n_neighbors=5 is standard starting point
}

# 5. Train, evaluate, store
model_data = {}
results = []

for name, model in models.items():
    model.fit(X_train_s, y_train)
    predictions = model.predict(X_test_s)
    probs       = model.predict_proba(X_test_s)[:, 1]

    acc     = accuracy_score(y_test, predictions)
    prec    = precision_score(y_test, predictions, zero_division=0)
    rec     = recall_score(y_test, predictions, zero_division=0)
    cm      = confusion_matrix(y_test, predictions)
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)

    results.append({"Model": name, "Accuracy": acc, "Precision (UP)": prec, "Recall (UP)": rec, "AUC": roc_auc})
    model_data[name] = {"predictions": predictions, "probs": probs, "cm": cm, "fpr": fpr, "tpr": tpr, "auc": roc_auc}
    print(f"  ✓ {name:<25} Acc={acc:.2%}  AUC={roc_auc:.3f}")

# 6. Leaderboard
leaderboard = pd.DataFrame(results).sort_values(by="AUC", ascending=False).reset_index(drop=True)
display(leaderboard.style.format({
    "Accuracy": "{:.2%}", "Precision (UP)": "{:.2f}", "Recall (UP)": "{:.2f}", "AUC": "{:.3f}"
}))

## Phase 5: Publication Graphs
Three figures are produced and saved at 300 DPI:
1. **Bar Chart** — Accuracy / Precision / Recall / AUC per model
2. **ROC Curves** — True vs False positive rates per model
3. **Confusion Matrices** — 5-panel grid, one per classifier

In [ ]:
print("\n============= PHASE 5: PUBLICATION GRAPHS =============")

# GRAPH 1: Performance comparison bar chart
plt.figure(figsize=(12, 6))
df_plot = leaderboard.melt(id_vars="Model", var_name="Metric", value_name="Score")
ax = sns.barplot(x="Model", y="Score", hue="Metric", data=df_plot, palette="viridis")
plt.title('Model Performance Comparison: Sentiment + Price Features', pad=15, fontweight='bold')
plt.ylim(0, 1.15)
plt.ylabel('Score (0.0 – 1.0)')
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
plt.tight_layout()
plt.savefig('Fig1_Model_Comparison.png', dpi=300)
plt.show()

# GRAPH 2: ROC Curves
plt.figure(figsize=(8, 8))
for name, data in model_data.items():
    plt.plot(data['fpr'], data['tpr'], lw=2, label=f"{name} (AUC = {data['auc']:.2f})")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve: Predicting GOOG Next-Day Movement', pad=15, fontweight='bold')
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig('Fig2_ROC_Curve.png', dpi=300)
plt.show()

# GRAPH 3: Confusion Matrices
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Confusion Matrices by Model', fontsize=16, fontweight='bold', y=1.02)
for ax, (name, data) in zip(axes, model_data.items()):
    sns.heatmap(data['cm'], annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'])
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('Fig3_Confusion_Matrices.png', bbox_inches='tight', dpi=300)
plt.show()

print("All graphs saved to directory.")